# Vectorless RAG: Retrieval-Augmented Generation Without Vector Databases

This Jupyter notebook explores vectorless RAG architectures - innovative approaches to retrieval-augmented generation that avoid traditional vector database dependencies. These methods leverage alternative indexing, search, and retrieval techniques to achieve effective document retrieval and question answering.

In [ ]:
# Setup: Load environment variables from .env file
import os
# Try to load dotenv, handle gracefully if not installed
try:
    from dotenv import load_dotenv
    load_dotenv()
    DOTENV_AVAILABLE = True
except ImportError:
    DOTENV_AVAILABLE = False
    print("Note: python-dotenv not installed. Using os.environ directly.")
    print("Install with: pip install python-dotenv")


# Retrieve API keys from environment
LANGCHAIN_API_KEY = os.getenv("LANGCHAIN_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
HUGGINGFACEHUB_API_TOKEN = os.getenv("HUGGINGFACEHUB_API_TOKEN")

print("Environment variables loaded:")
print(f"  LANGCHAIN_API_KEY: {'Set' if LANGCHAIN_API_KEY else 'Not set'}")
print(f"  OPENAI_API_KEY: {'Set' if OPENAI_API_KEY else 'Not set'}")
print(f"  HUGGINGFACEHUB_API_TOKEN: {'Set' if HUGGINGFACEHUB_API_TOKEN else 'Not set'}")



## Introduction to Vectorless RAG

**Vectorless RAG** refers to retrieval-augmented generation systems that operate without traditional vector databases and dense embeddings. These approaches are particularly valuable when:

- Memory or storage constraints prevent vector storage
- Real-time retrieval with low latency is critical
- Computational resources are limited
- The need for exact matches or BM25-style search is prioritized

**Traditional Vector RAG Limitations**:
- High memory footprint (storing millions of vectors)
- Complex indexing and similarity search infrastructure
- Embedding generation latency
- Search hallucination and noise issues

**Vectorless Advantages**:
- Lower memory requirements
- Simpler infrastructure
- Faster retrieval times
- More interpretable results


## 1. BM25 + LLM Architecture

**Description**: Combines classical BM25 search with language models, avoiding vector embeddings entirely.

**Key Components**:
- BM25 ranking algorithm
- Full-text search indexes
- LLM for context compression and generation
- Ranking optimization layers

**Implementation**:



In [ ]:
# BM25 + LLM Architecture
import os


class BM25RAG:
    def __init__(self, documents, llm):
        self.llm = llm
        self.bm25_index = self._build_bm25_index(documents)
    
    def _build_bm25_index(self, documents):
        # Build BM25 index using scikit-learn or custom implementation
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.metrics.pairwise import cosine_similarity
        
        vectorizer = TfidfVectorizer(
            max_features=50000,
            stop_words='english',
            ngram_range=(1, 3)
        )
        tfidf_matrix = vectorizer.fit_transform(documents)
        return {'vectors': tfidf_matrix, 'vocab': vectorizer.vocabulary_}
    
    def retrieve_and_generate(self, query, top_k=5):
        # BM25-style search using cosine similarity of TF-IDF vectors
        query_vec = self._text_to_vector(query)
        similarities = cosine_similarity(query_vec, self.bm25_index['vectors'])[0]
        top_indices = similarities.argsort()[-top_k:][::-1]
        
        # Retrieve documents
        retrieved_docs = [documents[i] for i in top_indices]
        
        # Context compression with LLM
        compressed_context = self._compress_context(query, retrieved_docs)
        
        # Generate response
        return self.llm.generate(f"Context:\n{compressed_context}\n\nQuestion: {query}")
    
    def _text_to_vector(self, text):
        # Simple TF-IDF vectorization for query
        vectorizer = TfidfVectorizer(**self.bm25_index.get('vectorizer_params', {}))
        return vectorizer.transform([text])
    
    def _compress_context(self, query, documents, max_tokens=1024):
        # Use LLM to select most relevant content
        context_text = "\n\n".join(documents)
        prompt = f"Given this query: '{query}', select the most relevant content from:\n\n{context_text}\n\nReturn only the most relevant content:"
        return self.llm.generate(prompt, max_length=max_tokens)



## 2. Tree-Structured RAG

**Description**: Hierarchical document organization with tree-based indexing for efficient search without vectors.

**Key Components**:
- Hierarchical document tree structure
- Recursive text chunking
- Tree traversal algorithms
- Cached search results

**Implementation**:



In [ ]:
# Tree-Structured RAG Implementation
import os


class TreeRAG:
    def __init__(self, documents, llm):
        self.llm = llm
        self.document_tree = self._build_document_tree(documents)
    
    def _build_document_tree(self, documents, max_branch_factor=10, max_depth=5):
        # Create hierarchical tree structure
        class TreeNode:
            def __init__(self, content):
                self.content = content
                self.children = []
        
        tree = TreeNode(content=documents[0])
        current_level = [tree]
        doc_index = 1
        
        for level in range(max_depth):
            if doc_index >= len(documents):
                break
            new_nodes = []
            for i in range(min(max_branch_factor, len(documents) - doc_index)):
                node = TreeNode(content=documents[doc_index + i])
                current_level[-1].children.append(node)
                new_nodes.append(node)
                doc_index += 1
            current_level = new_nodes
        
        return tree
    
    def retrieve_and_generate(self, query, top_k=5):
        # Tree-based search
        relevant_nodes = self._tree_search(query, self.document_tree, top_k)
        
        # Extract context from relevant nodes
        context = self._extract_context(relevant_nodes)
        
        # Generate response
        return self.llm.generate(f"Context:\n{context}\n\nQuestion: {query}")
    
    def _tree_search(self, query, node, top_k):
        # Recursively search tree for relevant content
        relevant_results = []
        
        # Search current node
        node_score = self._score_node(query, node)
        relevant_results.append((node_score, node))
        
        # Recursively search children
        for child in node.children:
            child_results = self._tree_search(query, child, top_k - len(relevant_results))
            relevant_results.extend(child_results)
        
        # Return top k results
        return sorted(relevant_results, key=lambda x: x[0], reverse=True)[:top_k]
    
    def _score_node(self, query, node):
        # Simple text matching score (vectorless approach)
        query_words = set(query.lower().split())
        node_words = set(node.content.lower().split())
        
        # Jaccard similarity
        intersection = query_words.intersection(node_words)
        union = query_words.union(node_words)
        return len(intersection) / len(union) if union else 0



## 3. Keyword-Graph RAG

**Description**: Uses keyword relationships and semantic graphs instead of vector similarity for retrieval.

**Key Components**:
- Keyword extraction and frequency analysis
- Semantic relationship mapping
- Graph-based connectivity analysis
- Path-based context propagation

**Implementation**:



In [ ]:
# Keyword-Graph RAG Implementation
import os


class KeywordGraphRAG:
    def __init__(self, documents, llm):
        self.llm = llm
        self.keyword_graph = self._build_keyword_graph(documents)
        self.semantic_index = self._build_semantic_index(documents)
    
    def _build_keyword_graph(self, documents):
        # Create graph of keyword relationships
        graph = {
            'keywords': set(),
            'relationships': {},
            'doc_keywords': {}  # document -> list of keywords
        }
        
        for i, doc in enumerate(documents):
            keywords = self._extract_keywords(doc)
            graph['doc_keywords'][i] = keywords
            graph['keywords'].update(keywords)
            
            for keyword in keywords:
                if keyword not in graph['relationships']:
                    graph['relationships'][keyword] = set()
        
        # Build relationships based on co-occurrence
        for doc_keywords in graph['doc_keywords'].values():
            for kw1 in doc_keywords:
                for kw2 in doc_keywords:
                    if kw1 != kw2:
                        graph['relationships'][kw1].add(kw2)
        
        return graph
    
    def _build_semantic_index(self, documents):
        # Create semantic index without vectors
        return {
            'inverted_index': self._build_inverted_index(documents),
            'doc_lengths': self._calculate_doc_lengths(documents),
            'total_terms': self._count_total_terms(documents)
        }
    
    def retrieve_and_generate(self, query, top_k=5):
        # Extract query keywords
        query_keywords = self._extract_keywords(query)
        
        # Find related documents through keyword graph
        related_docs = self._find_related_documents(query_keywords)
        
        # Rank documents using semantic index
        ranked_docs = self._rank_documents_semantic(query, related_docs)
        
        # Context extraction
        context = self._extract_context(ranked_docs[:top_k])
        
        # Generate response
        return self.llm.generate(f"Context:\n{context}\n\nQuestion: {query}")
    
    def _find_related_documents(self, keywords, max_depth=3):
        # Traverse keyword graph to find related documents
        visited_docs = set()
        queue = [(kw, 0) for kw in keywords]
        
        while queue:
            current_kw, depth = queue.pop(0)
            if depth >= max_depth:
                continue
                
            # Find documents containing this keyword
            for doc_idx, doc_keywords in self.keyword_graph['doc_keywords'].items():
                if doc_idx in visited_docs:
                    continue
                    if current_kw in doc_keywords:
                        visited_docs.add(doc_idx)
                        # Add related keywords for further exploration
                        related_kws = self.keyword_graph['relationships'].get(current_kw, set())
                        queue.extend([(kw, depth + 1) for kw in related_kws])
        
        return list(visited_docs)



## 4. Keyword-Index RAG

**Description**: Uses traditional inverted file indexes and keyword matching for retrieval.

**Key Components**:
- Inverted file index creation
- Positional indexing
- TF-IDF scoring
- Skip-list for efficient scanning

**Implementation**:



In [ ]:
# Keyword-Index RAG Implementation
import os


class KeywordIndexRAG:
    def __init__(self, documents, llm):
        self.llm = llm
        self.inverted_index = self._build_inverted_index(documents)
        self.doc_lengths = self._calculate_doc_lengths(documents)
        self.total_terms = len(documents)
    
    def _build_inverted_index(self, documents):
        # Create inverted index mapping keywords to documents
        inverted_index = {}
        term_doc_counts = {}
        
        for doc_id, doc in enumerate(documents):
            # Tokenize and normalize
            tokens = self._tokenize(doc)
            unique_terms = set(tokens)
            
            for term in unique_terms:
                if term not in inverted_index:
                    inverted_index[term] = []
                inverted_index[term].append(doc_id)
                
                if term not in term_doc_counts:
                    term_doc_counts[term] = 0
                term_doc_counts[term] += 1
        
        return inverted_index
    
    def _calculate_doc_lengths(self, documents):
        # Calculate document lengths for scoring
        return [len(self._tokenize(doc)) for doc in documents]
    
    def retrieve_and_generate(self, query, top_k=5):
        # Tokenize query
        query_tokens = self._tokenize(query)
        
        # Score documents using TF-IDF
        doc_scores = self._score_documents(query_tokens)
        
        # Get top k documents
        top_docs = self._get_top_documents(doc_scores, top_k)
        
        # Context extraction
        context = self._extract_context(top_docs)
        
        # Generate response
        return self.llm.generate(f"Context:\n{context}\n\nQuestion: {query}")
    
    def _score_documents(self, query_tokens):
        # Calculate TF-IDF scores for each document
        doc_scores = [0.0] * self.total_terms
        
        for term in query_tokens:
            if term in self.inverted_index:
                doc_ids = self.inverted_index[term]
                term_doc_count = len(doc_ids)
                
                for doc_id in doc_ids:
                    # Calculate TF-IDF contribution
                    tf = self._calculate_tf(term, doc_id)
                    idf = math.log(self.total_terms / term_doc_count)
                    doc_scores[doc_id] += tf * idf
        
        return doc_scores
    
    def _get_top_documents(self, doc_scores, top_k):
        # Return indices of top scoring documents
        indexed_scores = list(enumerate(doc_scores))
        indexed_scores.sort(key=lambda x: x[1], reverse=True)
        return [idx for idx, score in indexed_scores[:top_k]]



## 5. OCR-Based RAG

**Description**: Uses OCR and text recognition without vector databases, ideal for image-based documents.

**Key Components**:
- OCR engine integration
- Text extraction and preprocessing
- OCR result indexing
- Text-based search and retrieval

**Implementation**:



In [ ]:
# OCR-Based RAG Implementation
import os


class OCRRAG:
    def __init__(self, image_files, llm):
        self.llm = llm
        self.extracted_texts = self._extract_text_from_images(image_files)
        self.text_index = self._build_text_index(self.extracted_texts)
    
    def _extract_text_from_images(self, image_files):
        # Extract text using OCR (Tesseract, PaddleOCR, etc.)
        extracted_texts = []
        for image_file in image_files:
            text = self._ocr_engine.extract_text(image_file)
            extracted_texts.append({
                'file': image_file,
                'text': text,
                'metadata': self._extract_image_metadata(image_file)
            })
        return extracted_texts
    
    def _build_text_index(self, extracted_texts):
        # Create text-based index
        index = {}
        for i, item in enumerate(extracted_texts):
            # Tokenize and build inverted index
            tokens = self._tokenize(item['text'])
            for token in set(tokens):
                if token not in index:
                    index[token] = []
                index[token].append(i)
        return index
    
    def retrieve_and_generate(self, query, top_k=5):
        # Search extracted texts using keyword matching
        query_tokens = self._tokenize(query.lower())
        scored_results = self._score_texts(query_tokens)
        
        # Get top k results
        top_indices = [idx for idx, score in sorted(scored_results, key=lambda x: x[1], reverse=True)[:top_k]]

        # Extract context
        context = self._build_context([self.extracted_texts[i] for i in top_indices])
        
        # Generate response
        return self.llm.generate(f"Context:\n{context}\n\nQuestion: {query}")

    def _score_texts(self, query_tokens):
        # Score texts based on token matches
        scores = []
        for i, text_item in enumerate(self.extracted_texts):
            score = 0.0
            tokens = set(self._tokenize(text_item['text'].lower()))
            for token in query_tokens:
                if token in tokens:
                    score += self._calculate_token_weight(token, text_item)
            scores.append((i, score))
        return scores



## 6. Temporal RAG

**Description**: Uses temporal indexing and chronological ordering for retrieval without vectors.

**Key Components**:
- Timestamp extraction and indexing
- Chronological search algorithms
- Time-weighted relevance scoring
- Temporal context propagation

**Implementation**:



In [ ]:
# Temporal RAG Implementation
import os


class TemporalRAG:
    def __init__(self, documents, llm):
        self.llm = llm
        self.temporal_index = self._build_temporal_index(documents)
    
    def _build_temporal_index(self, documents):
        # Create temporal index mapping timestamps to documents
        temporal_index = {}
        for doc in documents:
            timestamp = self._extract_timestamp(doc)
            date_str = self._format_date(timestamp)

            if date_str not in temporal_index:
                temporal_index[date_str] = []
            temporal_index[date_str].append(doc)
        return temporal_index
    
    def retrieve_and_generate(self, query, time_window=None, top_k=5):
        # Get current timestamp for temporal context
        current_time = self._get_current_timestamp()

        # Temporal search
        relevant_docs = self._temporal_search(query, current_time, time_window)

        # Context extraction with temporal awareness
        context = self._extract_temporal_context(relevant_docs, current_time)

        # Generate response
        return self.llm.generate(f"Context:\n{context}\n\nQuestion: {query}")

    def _temporal_search(self, query, current_time, time_window):
        # Search documents within time window
        window_start = self._calculate_time_window_start(current_time, time_window)
        relevant_docs = []

        for date_str, docs in self.temporal_index.items():
            if date_str >= window_start and date_str <= current_time:
                relevant_docs.extend(docs)

        # Sort by relevance and recency
        return self._rank_temporal_results(query, relevant_docs)



## 7. Hash-Based RAG

**Description**: Uses content hashing and exact matching for retrieval without similarity search.

**Key Components**:
- Content hashing algorithms
- Exact match detection
- Chunk-based content storage
- Hash-based retrieval

**Implementation**:



In [ ]:
# Hash-Based RAG Implementation
import os


class HashBasedRAG:
    def __init__(self, documents, llm):
        self.llm = llm
        self.content_store = self._create_content_store(documents)
        self.hash_index = self._build_hash_index(documents)
    
    def _create_content_store(self, documents):
        # Store document content with chunking
        store = []
        for i, doc in enumerate(documents):
            chunks = self._chunk_document(doc, chunk_size=512)
            for chunk_id, chunk in enumerate(chunks):
                chunk_hash = self._calculate_content_hash(chunk)
                store.append({
                    'doc_id': i,
                    'chunk_id': chunk_id,
                    'hash': chunk_hash,
                    'content': chunk,
                    'metadata': self._extract_metadata(doc)
                })
        return store
    
    def _build_hash_index(self, documents):
        # Create hash-based index for fast lookup
        hash_index = {}
        for doc in documents:
            chunks = self._chunk_document(doc, chunk_size=512)
            for chunk in chunks:
                chunk_hash = self._calculate_content_hash(chunk)
                if chunk_hash not in hash_index:
                    hash_index[chunk_hash] = []
                hash_index[chunk_hash].append(doc)
        return hash_index
    
    def retrieve_and_generate(self, query, top_k=5):
        # Find exact matches for query chunks
        query_chunks = self._chunk_document(query, chunk_size=512)
        matched_chunks = []

        for query_chunk in query_chunks:
            query_hash = self._calculate_content_hash(query_chunk)
            if query_hash in self.hash_index:
                matched_chunks.extend(self.hash_index[query_hash])

        # Deduplicate and rank results
        unique_docs = list(set(matched_chunks))
        context = self._extract_context(unique_docs[:top_k])

        # Generate response
        return self.llm.generate(f"Context:\n{context}\n\nQuestion: {query}")



## 8. Similarity-Lookup RAG

**Description**: Uses multiple lightweight similarity algorithms instead of vector embeddings for retrieval.

**Key Components**:
- Simhash for near-duplicate detection
- Jaccard similarity for set-based matching
- Cosine similarity on term vectors
- Combination ranking algorithms

**Implementation**:



In [ ]:
# Similarity-Lookup RAG Implementation
import os


class SimilarityLookupRAG:
    def __init__(self, documents, llm):
        self.llm = llm
        self.document_index = self._build_simple_index(documents)
    
    def _build_simple_index(self, documents):
        # Create simple index with multiple similarity metrics
        index = {}
        for i, doc in enumerate(documents):
            tokens = self._tokenize(doc.lower())
            token_set = set(tokens)

            index[i] = {
                'tokens': tokens,
                'token_set': token_set,
                'length': len(tokens),
                'unique_tokens': len(token_set)
            }
        return index
    
    def retrieve_and_generate(self, query, top_k=5):
        # Calculate multiple similarity scores
        query_tokens = self._tokenize(query.lower())
        query_set = set(query_tokens)

        scored_results = []
        for doc_id, doc_data in self.document_index.items():
            # Calculate multiple similarity metrics
            jaccard_sim = self._jaccard_similarity(query_set, doc_data['token_set'])
            cosine_sim = self._cosine_similarity(query_tokens, doc_data['tokens'])
            tfidf_sim = self._tfidf_similarity(query_tokens, doc_data)

            # Combine scores
            combined_score = self._combine_similarity_scores([
                jaccard_sim, cosine_sim, tfidf_sim
            ])

            scored_results.append((doc_id, combined_score))

        # Get top k results
        top_docs = [doc_id for doc_id, score in sorted(scored_results, key=lambda x: x[1], reverse=True)[:top_k]]
        context = self._extract_context([self.document_index[doc_id]['content'] for doc_id in top_docs])

        # Generate response
        return self.llm.generate(f"Context:\n{context}\n\nQuestion: {query}")



## Vectorless RAG Selection Guide

**Choose Vectorless RAG when**:

**Resource Constraints**:
- Limited memory (MB instead of GB)
- Edge devices with restricted storage
- Cost-sensitive deployments

**Performance Requirements**:
- Sub-second retrieval latency
- Real-time applications
- Scalable to large document collections

**Data Characteristics**:
- Text-heavy documents
- Structured text data
- Image-based documents (OCR)
- Time-sensitive information

**Comparison with Vector RAG**:

**Advantages of Vectorless RAG**:
- Lower memory footprint
- Simpler infrastructure
- Faster implementation
- More interpretable results
- Better for exact keyword matching

**Trade-offs**:
- May have lower semantic understanding
- Less effective for paraphrased queries
- Limited handling of complex semantics

**Hybrid Approaches**:
Consider combining vectorless and vector-based methods:
- Layered retrieval: vectorless first, then vector refinement
- Ensemble methods: multiple approaches combined
- Adaptive selection based on query type


---

# How to Use This Notebook

## Prerequisites

```bash
# Install dependencies
pip install -r requirements.txt

# Optional: For full functionality with real LLMs
pip install langchain langchain-community langchain-huggingface
pip install sentence-transformers faiss-cpu
```

## Environment Setup

1. Copy `.env.example` to `.env`:
   ```bash
   cp .env.example .env
   ```

2. Edit `.env` and add your API keys:
   ```
   LANGCHAIN_API_KEY=your_key_here
   OPENAI_API_KEY=your_key_here
   HUGGINGFACEHUB_API_TOKEN=your_token_here
   ```

## Running the Notebook

1. Start Jupyter:
   ```bash
   jupyter notebook
   ```

2. Open this notebook (`vectorless_rag.ipynb`)

3. **Run cells in order** - The first code cell sets up all mock implementations and environment variables

4. Each subsequent code cell demonstrates a different architecture/approach

## What Each Section Does

- **Setup Cell**: Loads environment variables, creates mock classes for demonstration
- **Architecture Cells**: Each cell implements a different RAG approach with runnable code
- **Mock Mode**: By default, runs with mock implementations (no API keys needed)
- **Production Mode**: Install full dependencies and add API keys for real LLM integration

## Customization

To use with real models instead of mocks:

```python
# Replace mock embeddings with real ones
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Replace mock LLM with real one
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# Use real vector store
from langchain_community.vectorstores import FAISS
vector_store = FAISS.from_documents(documents, embeddings)
```

## Troubleshooting

- **ModuleNotFoundError**: Run `pip install -r requirements.txt`
- **No API keys**: The notebook runs in mock mode by default
- **Import errors**: Some cells use optional imports (networkx, langgraph, etc.) - install if needed

## Next Steps

- Experiment with different architectures
- Swap mock components for real implementations
- Add your own documents and test queries
- Compare retrieval quality across approaches
